# 04 - Label QA Review (pre-training gate)

Stage L3. Reviews every label run before training: per-class stats, mechanical
flags (per-class thresholds), sampled galleries, and filmstrips of the worst
sequences. Auto-discovers label runs under `labels/runs/*` and
`labels/clicks/*`; QA outputs go to `labels/qa/<run_id>/`.

**Fix loop:** flagged click sequences -> re-click in 03 (REDO_OBJECTS); flagged
batch sequences -> tune corridor/threshold in 02, delete their rows, rerun.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import random
import re

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vision_uss_research.labeling.boxes_io import (load_boxes_tolerant,
                                                   resolve_frame_path)
from vision_uss_research.qa.flags import FlagThresholds, compute_flags
from vision_uss_research.runs import finish_run, new_run_id, start_run
from vision_uss_research.sequences import frames_window_id

N_GALLERY_PER_CLASS = 12
SEED = 0
THRESHOLDS = FlagThresholds()
WINDOW_ROOT = ARTIFACTS / "frames" / frames_window_id(20, (0.30, 0.85))

# auto-discover label runs (or list dicts explicitly)
LABEL_SOURCES = "auto"
if LABEL_SOURCES == "auto":
    LABEL_SOURCES = []
    for base, gold in [(ARTIFACTS / "labels" / "runs", False),
                       (ARTIFACTS / "labels" / "clicks", True)]:
        for run_dir in sorted(base.glob("*")) if base.exists() else []:
            csv_path = run_dir / "propagated_boxes.csv"
            if csv_path.exists():
                LABEL_SOURCES.append({"name": run_dir.name, "boxes_csv": csv_path,
                                      "frames_root": WINDOW_ROOT, "gold": gold})
for s in LABEL_SOURCES:
    print(f"source: {s['name']} (gold={s['gold']})")

QA_DIR = ARTIFACTS / "labels" / "qa" / new_run_id("qa", "review")
start_run(QA_DIR, config={"thresholds": THRESHOLDS.__dict__,
                          "sources": [s["name"] for s in LABEL_SOURCES]},
          inputs=[s["name"] for s in LABEL_SOURCES], repo_root=REPO_DIR)

In [ ]:
parts = []
frames_root_by_source = {}
for source in LABEL_SOURCES:
    df = load_boxes_tolerant(Path(source["boxes_csv"]))
    df["source"] = source["name"]
    frames_root_by_source[source["name"]] = Path(source["frames_root"])
    parts.append(df)
    print(f"{source['name']}: {len(df)} rows, {df['sequence_id'].nunique()} sequences")

labels = pd.concat(parts, ignore_index=True)
labels["has_box"] = labels["x0"].notna()
boxed = labels[labels["has_box"]].copy()

def resolve_frame(row):
    return resolve_frame_path(row, frames_root_by_source[row["source"]])

stats = (labels.groupby(["target_object", "source"])
         .agg(sequences=("sequence_id", "nunique"), frames=("has_box", "size"),
              frames_with_box=("has_box", "sum")))
stats["coverage"] = (stats["frames_with_box"] / stats["frames"]).round(3)
box_stats = (boxed.groupby(["target_object", "source"])
             .agg(area_frac_median=("mask_area_frac", "median")).round(4))
display(stats.join(box_stats))

## Mechanical flags

In [ ]:
flags_df = compute_flags(boxed, THRESHOLDS)
flags_df.to_csv(QA_DIR / "flagged_labels.csv", index=False)
print(f"flagged: {len(flags_df)} of {len(boxed)} "
      f"({100 * len(flags_df) / max(len(boxed), 1):.1f}%)")
if len(flags_df):
    display(flags_df.groupby(["target_object", "flags"]).size().to_frame("n"))
    display(flags_df.groupby("sequence_id").size()
            .sort_values(ascending=False).head(10).to_frame("n_flags"))

## Galleries (random per class in green; flagged in red)

In [ ]:
def draw_box(img, row, color):
    x0, y0, x1, y1 = int(row["x0"]), int(row["y0"]), int(row["x1"]), int(row["y1"])
    cv2.rectangle(img, (x0, y0), (x1, y1), color, 3)
    cv2.putText(img, f"{row['target_object']} f{int(row['frame_idx'])}",
                (x0, max(y0 - 8, 16)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return img

def show_gallery(rows, title, color=(0, 255, 0), cols=4):
    items = []
    for _, row in rows.iterrows():
        p = resolve_frame(row)
        if p is None:
            continue
        items.append(draw_box(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB),
                              row, color))
    if not items:
        print(f"{title}: no renderable frames")
        return
    rows_n = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 3.4 * rows_n), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, img in zip(axes.flat, items):
        ax.imshow(img)
    fig.suptitle(title, fontsize=13)
    fig.savefig(QA_DIR / f"gallery_{re.sub(r'[^a-z0-9]+', '_', title.lower())}.jpg",
                dpi=90, bbox_inches="tight")
    plt.show()

rng = random.Random(SEED)
for obj, grp in boxed.groupby("target_object"):
    idx = rng.sample(range(len(grp)), min(N_GALLERY_PER_CLASS, len(grp)))
    show_gallery(grp.iloc[idx], f"{obj} - random sample ({len(grp)} boxes)")
if len(flags_df):
    for obj, grp in flags_df.groupby("target_object"):
        show_gallery(grp.head(N_GALLERY_PER_CLASS),
                     f"{obj} - FLAGGED ({len(grp)})", color=(255, 40, 40))

## Filmstrips (drift check: worst sequences + one random per class)

In [ ]:
def show_filmstrip(seq_id, cam):
    grp = boxed[(boxed["sequence_id"] == seq_id)
                & (boxed["camera"] == cam)].sort_values("frame_idx")
    items = []
    for _, row in grp.iterrows():
        p = resolve_frame(row)
        if p is None:
            continue
        img = draw_box(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB),
                       row, (0, 255, 0))
        items.append(cv2.resize(img, (320, 214)))
    if not items:
        return
    fig, ax = plt.subplots(figsize=(min(2.2 * len(items), 40), 3))
    ax.imshow(np.hstack(items))
    ax.axis("off")
    ax.set_title(f"{seq_id} ({grp['target_object'].iloc[0]}, {cam})", fontsize=10)
    plt.show()

shown = set()
if len(flags_df):
    for seq_id in (flags_df.groupby("sequence_id").size()
                   .sort_values(ascending=False).head(3).index):
        cam = flags_df[flags_df["sequence_id"] == seq_id]["camera"].iloc[0]
        show_filmstrip(seq_id, cam)
        shown.add(seq_id)
for obj, grp in boxed.groupby("target_object"):
    candidates = [s for s in grp["sequence_id"].unique() if s not in shown]
    if candidates:
        seq_id = rng.choice(sorted(candidates))
        show_filmstrip(seq_id, grp[grp["sequence_id"] == seq_id]["camera"].iloc[0])

## Verdict

In [ ]:
summary = stats.join(box_stats)
summary.to_csv(QA_DIR / "qa_class_summary.csv")
finish_run(QA_DIR, summary={
    "total_boxes": int(len(boxed)),
    "flagged": int(len(flags_df)),
    "flag_rate": round(len(flags_df) / max(len(boxed), 1), 4),
})
print(f"QA outputs: {QA_DIR}")
print("clean -> notebook 05 (dataset build); fixable -> 03 REDO_OBJECTS / 02 rerun")